<a href="https://colab.research.google.com/github/abbibr/AI_ML_AMITY-Final-Project/blob/master/final_twitter_sentiment_analysis_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Initial data transformation

As an initial approach, all the main libraries and functions were summarized in the following cell, focusing on data visualization, text analysis, text vectorization, and model building.

Additionally, the stopwords from English were downloaded from the `nltk` library.

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing
pd.options.mode.chained_assignment = None
import os

# Google Colab uchun to'g'irlangan yo'l
base_path = '/content'

for dirname, _, filenames in os.walk(base_path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

from wordcloud import WordCloud #Word visualization
import matplotlib.pyplot as plt #Plotting properties
import seaborn as sns #Plotting properties
from sklearn.feature_extraction.text import CountVectorizer #Data transformation
from sklearn.model_selection import train_test_split #Data testing
from sklearn.linear_model import LogisticRegression #Prediction Model
from sklearn.metrics import accuracy_score #Comparison between real and predicted
from sklearn.preprocessing import LabelEncoder
import re #Regular expressions
import nltk
from nltk import word_tokenize
nltk.download('stopwords')

Then, the validation and train datasets were saved on two variables by using the function of `read_csv` from pandas, where both didn't have a data header.

In [ ]:
import zipfile

# ZIP faylni ochish
with zipfile.ZipFile('/content/twitter_training.csv.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

# Fayllarni o'qish
train = pd.read_csv('/content/twitter_training.csv')
val = pd.read_csv('/content/twitter_validation.csv')

print("Training data shakli:", train.shape)
print("Validation data shakli:", val.shape)
print("\nTraining data:")
print(train.head())

Later, the columns were renamed to represent the given data of tweets. But, with the first 5 rows analysis, it was recognized that positive sentiment was assigned to a "kill" thread related to a videogame. Even with this in consideration, the modeling, in this case, will the same as a traditional NLP project.

In [ ]:
train.columns=['id','information','type','text']
train.head()

Then, with the validation data, the information of the first 5 rows didn't show any unusual labeling.

In [ ]:
val.columns=['id','information','type','text']
val.head()

In [ ]:
train_data=train
train_data

In [ ]:
val_data=val
val_data

To prepare the data for the text analysis an additional row was created using the method of `str.lower`. However, as there were some texts with only numerical values (such as one that only had a 2 as the tweet) an additional function was used for transforming all the data to string.

Then, a regex expression erased the special characters as it is common to have digitation problems on Twitter.

In [ ]:
#Text transformation
train_data["lower"]=train_data.text.str.lower() #lowercase
train_data["lower"]=[str(data) for data in train_data.lower] #converting all to string
train_data["lower"]=train_data.lower.apply(lambda x: re.sub('[^A-Za-z0-9 ]+', ' ', x)) #regex
val_data["lower"]=val_data.text.str.lower() #lowercase
val_data["lower"]=[str(data) for data in val_data.lower] #converting all to string
val_data["lower"]=val_data.lower.apply(lambda x: re.sub('[^A-Za-z0-9 ]+', ' ', x)) #regex

The differences between the two text columns are presented in the next table.

In [ ]:
train_data.head()

# 2. Plotting features

As to identify the main words that were used per label, a word_cloud was used to see which are the most important words on the train data. For example, on the positive label words such as love and game were mostly used alongside a wide variety of words classified as "good sentiments".

In [ ]:
word_cloud_text = ''.join(train_data[train_data["type"]=="Positive"].lower)
#Creation of wordcloud
wordcloud = WordCloud(
    max_font_size=100,
    max_words=100,
    background_color="black",
    scale=10,
    width=800,
    height=800
).generate(word_cloud_text)
#Figure properties
plt.figure(figsize=(10,10))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()

As for the negative tweets, some curse words were the most important while the names of some games and industries were also very used, such as facebook and eamaddennfl.

In [ ]:
word_cloud_text = ''.join(train_data[train_data["type"]=="Negative"].lower)
#Creation of wordcloud
wordcloud = WordCloud(
    max_font_size=100,
    max_words=100,
    background_color="black",
    scale=10,
    width=800,
    height=800
).generate(word_cloud_text)
#Figure properties
plt.figure(figsize=(10,10))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()

The irrelevant tweets show a similar trend as negative ones, something that will impact the overall prediction performance.

In [ ]:
word_cloud_text = ''.join(train_data[train_data["type"]=="Irrelevant"].lower)
#Creation of wordcloud
wordcloud = WordCloud(
    max_font_size=100,
    max_words=100,
    background_color="black",
    scale=10,
    width=800,
    height=800
).generate(word_cloud_text)
#Figure properties
plt.figure(figsize=(10,10))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()

Then, on the neutral side, there are almost no curse words and the most important ones are different from the other 3 categories.

In [ ]:
word_cloud_text = ''.join(train_data[train_data["type"]=="Neutral"].lower)
#Creation of wordcloud
wordcloud = WordCloud(
    max_font_size=100,
    max_words=100,
    background_color="black",
    scale=10,
    width=800,
    height=800
).generate(word_cloud_text)
#Figure properties
plt.figure(figsize=(10,10))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()

Finally, in this section, the information was grouped by the brand (or in this case the column information) to make a barplot that shows the number of tweets for each one.

In [ ]:
#Count information per category
plot1=train.groupby(by=["information","type"]).count().reset_index()
plot1.head()

As an interesting fact, the number of modified texts coincides with the id. For this reason, as the ID is unique, the following barplot shows that for games such as MaddenNFL and NBA2K the number of negative tweets is the highest while on the other brands the trend is different.

In [ ]:
#Figure of comparison per branch
plt.figure(figsize=(20,6))
sns.barplot(data=plot1,x="information",y="id",hue="type")
plt.xticks(rotation=90)
plt.xlabel("Brand")
plt.ylabel("Number of tweets")
plt.grid()
plt.title("Distribution of tweets per Branch and Type");

# 3. Text analysis

With the clean text, the initial number of unique tokens was counted to identify the model complexity. As presented, there are more than 30 thousand unique words.

In [ ]:
#Text splitting
nltk.download('punkt')
nltk.download('punkt_tab')
tokens_text = [word_tokenize(str(word)) for word in train_data.lower]
#Unique word counter
tokens_counter = [item for sublist in tokens_text for item in sublist]
print("Number of tokens: ", len(set(tokens_counter)))

The tokens_text variable groups all the texts by the different words stored on a List.

In [ ]:
tokens_text[1]

Also, the main English stopwords were saved on an additional variable, to be used in the following modeling.

In [ ]:
#Choosing english stopwords
stopwords_nltk = nltk.corpus.stopwords
stop_words = stopwords_nltk.words('english')
stop_words[:5]

# 4. Logistic Regression model

For the main regression model, it was used a simple Logistic Regression of the sklearn library alongside the Bag of Words (BoW) approach. This last method helps to classify and group the relevant data to help the model identify the proper trends.

On this first BoW, the stopwords were considered alongside a default [ngram](https://deepai.org/machine-learning-glossary-and-terms/n-gram) of 1.

![Ngram.png](attachment:dbc1ae72-bb04-42d7-8509-1e7995069310.png)

In [ ]:
#Initial Bag of Words
bow_counts = CountVectorizer(
    tokenizer=word_tokenize,
    stop_words=stop_words, #English Stopwords
    ngram_range=(1, 1) #analysis of one word
)

Then, the main data was split on train and test datasets alongside the encoding of the words by using the training dataset as a reference:

In [ ]:
#Train - Test splitting
reviews_train, reviews_test = train_test_split(train_data, test_size=0.2, random_state=0)

In [ ]:
#Creation of encoding related to train dataset
X_train_bow = bow_counts.fit_transform(reviews_train.lower)
#Transformation of test dataset with train encoding
X_test_bow = bow_counts.transform(reviews_test.lower)

In [ ]:
X_test_bow

In [ ]:
#Labels for train and test encoding
y_train_bow = reviews_train['type']
y_test_bow = reviews_test['type']

The total number of tweets for each category shows that negative and positive are the most registered while the irrelevant is the lowest.

In [ ]:
#Total of registers per category
y_test_bow.value_counts() / y_test_bow.shape[0]

With this data, the Logistic Regression Model was trained,
achieving approximately 81% test accuracy and 93% validation
accuracy. The higher validation score reflects the small
size of the validation set (1,000 tweets), which makes
it less statistically reliable than the 15,000-row test set.

In [ ]:
# Logistic regression
model1 = LogisticRegression(C=1, solver='lbfgs', max_iter=200)
model1.fit(X_train_bow, y_train_bow)
# Prediction
test_pred = model1.predict(X_test_bow)
print("Accuracy: ", accuracy_score(y_test_bow, test_pred) * 100)

In [ ]:
#Validation data
X_val_bow = bow_counts.transform(val_data.lower)
y_val_bow = val_data['type']

In [ ]:
X_val_bow

In [ ]:
Val_res = model1.predict(X_val_bow)
print("Accuracy: ", accuracy_score(y_val_bow, Val_res) * 100)

A TF-IDF vectorizer was used with bigrams (ngram_range=(1,2))
and capped at 5,000 features to prevent overfitting.
Unlike raw word counts, TF-IDF down-weights common words
and up-weights rare, distinctive sentiment words.

The Test dataset got to 68% while on the validation data the accuracy was 78%, showing that this approach was better than the simple n-gram and stopwords model.

In [ ]:
from sklearn.feature_extraction.text \
    import TfidfVectorizer

tfidf_phase2 = TfidfVectorizer(
    max_features=5000,
    sublinear_tf=True,
    ngram_range=(1, 2)
)
X_train = tfidf_phase2.fit_transform(
    reviews_train.lower)
X_test  = tfidf_phase2.transform(
    reviews_test.lower)
X_val   = tfidf_phase2.transform(
    val_data.lower)

In [ ]:
X_train_bow

In [ ]:
model2 = LogisticRegression(C=0.9, solver='lbfgs', max_iter=1500)
model2.fit(X_train, reviews_train['type'])

test_pred_2 = model2.predict(X_test)
print("Test Accuracy: ", accuracy_score(y_test_bow, test_pred_2) * 100)

In [ ]:
y_val_bow = val_data['type']
Val_pred_2 = model2.predict(X_val)
print("Validation Accuracy: ", accuracy_score(y_val_bow, Val_pred_2) * 100)

In [ ]:
# Save model2 and its matching vectorizer as a paired set
import pickle
from google.colab import files

with open('sentiment_model.pkl', 'wb') as f:
    pickle.dump(model2, f)

with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_phase2, f)   # saves the matching vectorizer

files.download('sentiment_model.pkl')
files.download('vectorizer.pkl')

To optimize our Logistic Regression model, the best approach is to implement Regularization (L1 & L2) and compare it with a Support Vector Machine (SVM).


In Logistic Regression, "optimization" usually refers to finding the best hyperparameters (like the penalty type and the strength of regularization $C$) to prevent overfitting.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import pandas as pd

This code trains three versions: Logistic Regression with L2 (Ridge), L1 (Lasso), and a Linear SVM. It also calculates the F1-Score, which is more reliable than accuracy if your tweet sentiments are imbalanced.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# 1. LOAD DATA
df = pd.read_csv('twitter_training.csv', header=None)
df.columns = ['id', 'entity', 'sentiment', 'text']
df.dropna(subset=['text'], inplace=True)

# 2. SPLIT RAW TEXT FIRST (before any vectorization)
X_raw = df['text'].astype(str)
y = df['sentiment']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# 3. VECTORIZATION — fit ONLY on train, transform both
# This prevents data leakage from test set into IDF weights
tfidf = TfidfVectorizer(
    max_features=5000,
    sublinear_tf=True,       # improves performance on text
    ngram_range=(1, 2)       # captures bigrams like "not good"
)
X_train = tfidf.fit_transform(X_train_raw)   # fit + transform on train only
X_test  = tfidf.transform(X_test_raw)        # transform only — NO fit

# 4. TRAIN MODELS
log_l2 = LogisticRegression(penalty='l2', C=1.0, max_iter=1000)
log_l2.fit(X_train, y_train)

log_l1 = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, max_iter=1000)
log_l1.fit(X_train, y_train)

# Wrap LinearSVC to add probability support
base_svm = LinearSVC(C=1.0, max_iter=1000)
svm_model = CalibratedClassifierCV(base_svm)
svm_model.fit(X_train, y_train)

# 5. EVALUATE
models = ['LogReg L2', 'LogReg L1', 'SVM']
preds  = [log_l2.predict(X_test),
          log_l1.predict(X_test),
          svm_model.predict(X_test)]

results = pd.DataFrame({
    'Model'   : models,
    'Accuracy': [accuracy_score(y_test, p) for p in preds],
    'F1-Score': [f1_score(y_test, p, average='weighted') for p in preds]
})

print(results)

# 6. TRAIN vs TEST gap check (overfitting diagnosis)
train_preds = [log_l2.predict(X_train),
               log_l1.predict(X_train),
               svm_model.predict(X_train)]

results['Train Accuracy'] = [accuracy_score(y_train, p) for p in train_preds]
results['Gap'] = results['Train Accuracy'] - results['Accuracy']
print("\nOverfitting check:")
print(results[['Model', 'Train Accuracy', 'Accuracy', 'Gap']])

Class Frequency Distribution. This bar chart quantifies the absolute volume of data points available for each sentiment category. By visualizing the raw counts, we can identify if any specific sentiment is underrepresented, which is critical for determining whether data augmentation or specific evaluation metrics (like F1-Score) are required.

While the Bar Chart provides the exact sample size necessary for technical validation, the Pie Chart offers a simplified perspective on the overall 'mood' of the dataset, making the findings accessible to both technical and non-technical stakeholders.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Prepare data for Seaborn (Melting the dataframe)
# This assumes we have the 'results' dataframe from the previous step
df_viz = results.melt(id_vars='Model', var_name='Metric', value_name='Score')

# 2. Set the aesthetic style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# 3. Create a Grouped Bar Chart
ax = sns.barplot(data=df_viz, x='Model', y='Score', hue='Metric', palette='magma')

# Add labels and title
plt.title('Model Optimization Comparison: Accuracy vs F1-Score', fontsize=16, fontweight='bold', pad=20)
plt.ylim(0, 1.0) # Scores are between 0 and 1
plt.ylabel('Score Value', fontsize=12)
plt.xlabel('Algorithm Version', fontsize=12)

# Annotate bars with the actual values
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.2f'),
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha = 'center', va = 'center',
                xytext = (0, 9),
                textcoords = 'offset points',
                fontsize=10, fontweight='bold')

plt.legend(title='Metric', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# 4. Create a separate Line Plot for Trend
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_viz, x='Model', y='Score', hue='Metric', marker='o', linewidth=2.5)
plt.title('Performance Trend Across Model Variations', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
import pandas as pd
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. SETUP: Loading specific values
try:
    column_names = ['ID', 'Entity', 'Sentiment', 'Content']
    df = pd.read_csv('twitter_training.csv', names=column_names)
    df.dropna(subset=['Content'], inplace=True)

    model = joblib.load('sentiment_model.pkl')
    vectorizer = joblib.load('vectorizer.pkl')

    # Generate decision scores (z) for your actual data
    X = vectorizer.transform(df['Content'].astype(str))
    # Get the actual probabilities (The S-Curve values)
    probabilities = model.predict_proba(X)
    # We use decision_function to get the linear combination (z) before the Sigmoid
    z_scores = np.log(probabilities / (1 - probabilities + 1e-10))

    print("Success: Your model values have been loaded!")
except Exception as e:
    print(f"Error: Make sure twitter_training.csv, sentiment_model.pkl, and vectorizer.pkl are uploaded. \n{e}")


# 2. VISUALIZATION: The Empirical Sigmoid "Regression Line"
# This shows how model turns tweets into probabilities
plt.figure(figsize=(10, 6))
# We plot the 'z' score vs the 'Positive' probability (usually index 3)
# Taking a subset for clear visualization
pos_index = list(model.classes_).index('Positive')
z_sample = z_scores[:1000, pos_index]
p_sample = probabilities[:1000, pos_index]

plt.scatter(z_sample, p_sample, color='#3498db', alpha=0.3, s=10, label='Actual Tweet Predictions')
plt.axhline(0.5, color='#e74c3c', linestyle='--', label='Decision Threshold')
plt.title('Figure 4.8: Empirical Logistic Regression Curve (Sigmoid)', fontsize=14)
plt.xlabel('Linear Combination (Weighted Word Scores)', fontsize=12)
plt.ylabel('Probability of Positive Sentiment', fontsize=12)
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

# 3. VISUALIZATION: L2 Regularization Effect
# This shows how L2 kept your word weights small and stable
all_weights = model.coef_.flatten()

plt.figure(figsize=(10, 6))
sns.histplot(all_weights, bins=100, color='#2ecc71', kde=True)
plt.axvline(0, color='black', linewidth=1)
plt.title('Figure 4.9: Impact of L2 Regularization on Feature Weights', fontsize=14)
plt.xlabel('Magnitude of Word Coefficients', fontsize=12)
plt.ylabel('Frequency (Number of Words)', fontsize=12)
plt.yscale('log') # Log scale helps see the distribution clearly
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# column_names = ['ID', 'Entity', 'Sentiment', 'Content']
# df = pd.read_csv('twitter_training.csv', names=column_names)

# Bar Chart
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Sentiment', palette='magma')
plt.title('Figure 4.1: Frequency of Sentiment Classes', fontsize=14)
plt.xlabel('Sentiment Label')
plt.ylabel('Number of Samples')
plt.show()

# Pie Chart
plt.figure(figsize=(6, 6))
df['Sentiment'].value_counts().plot.pie(autopct='%1.1f%%', colors=sns.color_palette('pastel'), startangle=140)
plt.title('Figure 4.2: Proportional Share of Twitter Sentiments')
plt.ylabel('')
plt.show()


In [ ]:
y_pred_l2 = preds[0]
y_pred_l1 = preds[1]
y_pred_svm = preds[2]

# Calculating Test and train results
train_acc = [accuracy_score(y_train, log_l2.predict(X_train)),
             accuracy_score(y_train, log_l1.predict(X_train)),
             accuracy_score(y_train, svm_model.predict(X_train))]

test_acc = [accuracy_score(y_test, y_pred_l2),
            accuracy_score(y_test, y_pred_l1),
            accuracy_score(y_test, y_pred_svm)]

# Show in the graph
models = ['LogReg L2', 'LogReg L1', 'SVM']
x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x - width/2, train_acc, width, label='Train Accuracy', color='#3498db')
plt.bar(x + width/2, test_acc, width, label='Test Accuracy ', color='#e74c3c')

plt.ylabel('Accuracy')
plt.title('Train vs Test Accuracy Comparison')
plt.xticks(x, models)
plt.legend()
plt.ylim(0, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Here is the graph of using only SVM algorithm.

Support Vector Machines (SVM) are particularly effective for high-dimensional data, such as text represented by TF-IDF vectors, because they operate on the principle of Structural Risk Minimization. Unlike models that focus on overall probability, SVM seeks the Maximum Margin Hyperplane—the boundary that provides the largest possible distance between different sentiment classes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample

# Rebuild to guarantee they match
X_train_pca_source = tfidf.transform(X_train_raw)   # uses Phase 3 raw text
X_sample, y_sample = resample(X_train_pca_source, y_train,
                               n_samples=3000,
                               random_state=42)

# 1. Dimensionality Reduction
# Since we have thousands of words (features), we use PCA to reduce them to 2 dimensions for visualization
# PCA
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_sample.toarray())
# Train SVM on 2D data
svm_2d = LinearSVC(C=1.0, max_iter=10000)
svm_2d.fit(X_train_pca, y_sample)

# 3. Define the mesh grid for the plot background
# This creates a grid of points to color the decision regions
h = .05  # Step size in the mesh
x_min, x_max = X_train_pca[:, 0].min() - 0.5, X_train_pca[:, 0].max() + 0.5
y_min, y_max = X_train_pca[:, 1].min() - 0.5, X_train_pca[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# 4. Predict classifications for every point in the mesh grid
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()])

# 5. Encode labels to numerical values for plotting colors
le = LabelEncoder()
y_encoded = le.fit_transform(y_sample)
Z_encoded = le.transform(Z)
Z_encoded = Z_encoded.reshape(xx.shape)

# 6. Visualization
plt.figure(figsize=(10, 7))

# Plot the decision boundaries (the background colors)
plt.contourf(xx, yy, Z_encoded, cmap=plt.cm.coolwarm, alpha=0.2)

# Plot the actual data points from the training set
scatter = plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_encoded,
            cmap=plt.cm.coolwarm, edgecolors='k', s=40, alpha=0.8)

# Add titles and axis labels
plt.title('Figure 4.10: SVM Decision Boundary (2D PCA Projection)', fontsize=14, fontweight='bold')
plt.xlabel('Principal Component 1 (PCA1)')
plt.ylabel('Principal Component 2 (PCA2)')

# Add a legend to identify each sentiment class
plt.legend(handles=scatter.legend_elements()[0], labels=list(le.classes_), loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

Real-Time Sentiment Prediction

In [ ]:
def predict_my_tweet(text, model, vectorizer):
    # 1. Transform the input text into a numerical format using the vectorizer
    text_vectorized = vectorizer.transform([text])

    # 2. Perform the prediction using the trained model
    prediction = model.predict(text_vectorized)[0]

    # 3. Calculate the probability scores for each class
    probabilities = model.predict_proba(text_vectorized)[0]
    # Identify the highest probability percentage
    max_prob = max(probabilities) * 100

    print(f"--- Prediction Results ---")
    print(f"Input text: '{text}'")
    print(f"Sentiment Label: {prediction.upper()}")
    print(f"Confidence Level: {max_prob:.2f}%")
    print("-" * 25)

# --- MODEL TESTING ---
# You can enter any custom sentence inside the quotes below:
my_text = "It is a cool, but you know sometimes it is not working quite good. So, I guess it depends on the situation..."

# Select the best performing model for testing (e.g., log_l2 or svm_model)
predict_my_tweet(my_text, log_l2, tfidf)


def predict_my_tweet_svm(text, model, vectorizer):
    # 1. Vectorize text
    text_vectorized = vectorizer.transform([text])

    # 2. Get the label
    prediction = model.predict(text_vectorized)[0]

    # 3. Get probabilities (Right now, available via CalibratedClassifierCV for SVM)
    # This replaces the decision_function logic
    probabilities = model.predict_proba(text_vectorized)[0]
    confidence = max(probabilities) * 100

    print(f"--- SVM Prediction Results (Calibrated) ---")
    print(f"Input text: '{text}'")
    print(f"Sentiment Label: {prediction.upper()}")
    print(f"Confidence Level: {confidence:.2f}%")
    print("-" * 35)

# Test it
predict_my_tweet_svm("It is a cool, but you know sometimes it is not working quite good. So, I guess it depends on the situation...", svm_model, tfidf)